# SQL Layer: Queries and Power BI Exports

This notebook runs the SQL files in `../sql/` against the SQLite database and exports the results for the Power BI dashboard.

**Run `01_ab_test_analysis.ipynb` first.** It creates the database (`data/cookie_cats.db`) and the `analysis_params` table used by `kpi_summary.sql`.

| SQL file | Purpose | Export |
|---|---|---|
| `player_journey.sql` | Installed, returned on day 1, returned on day 7 | `player_journey.csv` |
| `kpi_summary.sql` | Players, D1/D7 retention, average rounds | `kpi_summary.csv` |
| `round_distribution.sql` | Share of players per rounds-played bucket | `round_distribution.csv` |

The last section reconciles the SQL results with pandas, so the dashboard and the notebook report identical numbers.

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
SQL_DIR = Path("../sql")
EXPORT_DIR = DATA_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)

conn = sqlite3.connect(DATA_DIR / "cookie_cats.db")


def run_sql_file(filename):
    """Run a query stored in the sql/ folder and return the result as a DataFrame."""
    query = (SQL_DIR / filename).read_text(encoding="utf-8")
    return pd.read_sql(query, conn)

## Sanity check

In [2]:
pd.read_sql("SELECT COUNT(*) AS players FROM cookie_cats", conn)

,players
0,90189


## 1. Player journey

Installed, returned on day 1, returned on day 7. The result is reshaped to long format, which is what the Power BI funnel chart expects.

In [3]:
player_journey_wide = run_sql_file("player_journey.sql")
player_journey_wide

,version,installed,returned_day1,returned_day7
0,gate_30,44700,20034,8502
1,gate_40,45489,20119,8279


In [4]:
stage_order = {"installed": 1, "returned_day1": 2, "returned_day7": 3}

player_journey = (
    player_journey_wide
    .melt(id_vars="version", var_name="stage", value_name="players")
    .assign(stage_order=lambda d: d["stage"].map(stage_order))
    .sort_values(["version", "stage_order"])
    .reset_index(drop=True)
)
player_journey.to_csv(EXPORT_DIR / "player_journey.csv", index=False)
player_journey

,version,stage,players,stage_order
0,gate_30,installed,44700,1
1,gate_30,returned_day1,20034,2
2,gate_30,returned_day7,8502,3
3,gate_40,installed,45489,1
4,gate_40,returned_day1,20119,2
5,gate_40,returned_day7,8279,3


## 2. KPI summary

In [8]:
kpi_summary = run_sql_file("kpi_summary.sql")
kpi_summary.to_csv(EXPORT_DIR / "kpi_summary.csv", index=False)
kpi_summary

,version,total_players,d1_retention_pct,d7_retention_pct,avg_rounds_capped
0,gate_30,44700,44.82,19.02,51.2
1,gate_40,45489,44.23,18.20,51.1


## 3. Round distribution

In [6]:
round_distribution = run_sql_file("round_distribution.sql")
round_distribution.to_csv(EXPORT_DIR / "round_distribution.csv", index=False)
round_distribution

,version,round_bucket,bucket_order,player_count,pct_of_version
0,gate_30,0,1,1937,4.33
1,gate_30,1-10,2,15736,35.20
2,gate_30,11-50,3,15891,35.55
3,gate_30,51-100,4,5041,11.28
4,gate_30,100+,5,6095,13.64
5,gate_40,0,1,2057,4.52
6,gate_40,1-10,2,16259,35.74
7,gate_40,11-50,3,15497,34.07
8,gate_40,51-100,4,5386,11.84
9,gate_40,100+,5,6290,13.83


## 4. Reconciliation with pandas

The SQL results must match a direct pandas calculation on the same table.

In [7]:
raw = pd.read_sql("SELECT * FROM cookie_cats", conn)

pandas_check = raw.groupby("version").agg(
    n=("userid", "count"),
    d1=("retention_1", "mean"),
    d7=("retention_7", "mean"),
)

merged = kpi_summary.set_index("version").join(pandas_check)

assert (merged["total_players"] == merged["n"]).all()
assert np.allclose(merged["d1_retention_pct"], merged["d1"] * 100, atol=0.01)
assert np.allclose(merged["d7_retention_pct"], merged["d7"] * 100, atol=0.01)
assert (
    player_journey_wide.set_index("version")["installed"] == merged["total_players"]
).all()

print("SQL results reconcile with pandas.")

SQL results reconcile with pandas.


In [ ]:
conn.close()